# 2.1 · PD fine-tuning

**How every Experiment 2 configuration behaved *while it trained*** — the loss, the real-data AUC, out-of-domain retention, every logged metric, the prior levers pulled apart, per-dataset behaviour, hardware and the gradient signal. Reads the per-arm `output/manifests/exp2_pd__*__progress.csv` and `__telemetry.csv`, so it works on a **partial sweep** and re-runs cleanly as more arms finish.

**What Experiment 2 is.** Exp2 **warm-starts the released TabICLv2 weights** and continues training on a mixture of the original prior and ours, sweeping `credit_fraction × init.strategy × l2sp_alpha × lr` (60 arms). The question is not only whether credit transfers but at what **out-of-domain cost** — Tanna 2026 reports naive full fine-tuning collapses TabICL (TabZilla 0.873→0.567), which is exactly why freeze depth and L2-SP are swept (`papers/2026/04_Tanna_ExploringFineTuning`; recipe from Kolberg TabPFN-Wide 2026).

*Grounding.* Figures overlay published values as teal reference lines where the literature gives one, cited by path against `tfm-library` pin `52dab01`. Interpretation lives in this prose and the captions; the figures carry data and short labels only.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.visualize import training_plots, figures, style, literature

style.apply()   # ONE shared style: identical colours in every figure of every notebook
pd.set_option("display.width", 200, "display.max_columns", 40)

TASK = "pd"
EXP = "exp2"
# Every started arm leaves output/manifests/exp2_pd__<run>__progress.csv and __telemetry.csv;
# these read them, so a PARTIAL sweep still plots. Clears THIS notebook's figure folder first.
FIGS = figures.FigureSaver("2.1_pd_finetuning")

## 0. The colour key

In [ ]:
FIGS.save(style.show_palette(), "palette",
    caption="The shared colour vocabulary used on every axis of this notebook: each swatch names the prior, data source, literature overlay or annotation it marks.");

## 1. Training loss

The loss is the only thing every arm optimises directly; before reading any real-data metric, a reader checks here that every arm actually descended and none diverged. TabICLv2's own ablations warn of a prior×architecture interaction that can make training diverge on the wrong prior (`papers/2026/02_Qu_TabICLv2 §4.4`), so divergence here is a **reportable outcome**, not something to tune away.

In [ ]:
FIGS.save(training_plots.training_loss(TASK, exp=EXP), "training_loss",
    caption="Training loss against optimisation step for every PD arm (grey), the mean across arms (black), and the best and worst arms by final real-data AUC (highlighted).");

## 2. Real-data AUC over training

The loss is measured on synthetic data; this is the first sight of the model on the **real** credit datasets it never trained on, scored every `progress.every_datasets` steps. A prior that helps should lift this curve earlier or higher than the control.

In [ ]:
FIGS.save(training_plots.metric_over_training(TASK, exp=EXP), "metric_over_training",
    caption="Real-data AUC, averaged over the evaluation datasets, against training step; one line per arm coloured by prior (credit versus control), with the across-arm mean in black.");

## 3. Credit prior versus control

The headline Experiment comparison drawn as a curve: the credit-prior arms against the `credit_fraction=0` control that is, by construction, exactly TabICLv2's own prior. The median and inter-quartile band make the seed spread visible — arms are never ranked on a gap smaller than that spread.

In [ ]:
FIGS.save(training_plots.credit_vs_control_over_training(TASK, exp=EXP), "credit_vs_control_over_training",
    caption="Real-data AUC against training step for credit-prior arms versus control arms; bold median lines with shaded inter-quartile bands across the arms in each group.");

## 4. AUC by freeze strategy

Isolating one lever: all arms sharing a value of **freeze strategy** averaged together, so the reader sees whether that knob alone moves the curve. Freeze depth is the lever against catastrophic forgetting — TabICL collapses under naive full fine-tuning (`papers/2026/04_Tanna_ExploringFineTuning`).

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "strategy", exp=EXP), "metric_by_lever_strategy",
    caption="Real-data AUC against training step, one mean line per value of freeze strategy, averaged over the arms that share each value.");

## 5. AUC by L2-SP

Isolating one lever: all arms sharing a value of **L2-SP** averaged together, so the reader sees whether that knob alone moves the curve. L2-SP pulls the weights toward the released checkpoint; 0.003 is Real-TabPFN's value (`papers/2025/07_Garg_RealTabPFN`), swept here against off.

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "l2sp", exp=EXP), "metric_by_lever_l2sp",
    caption="Real-data AUC against training step, one mean line per value of L2-SP, averaged over the arms that share each value.");

## 6. AUC by credit fraction

Isolating one lever: all arms sharing a value of **credit fraction** averaged together, so the reader sees whether that knob alone moves the curve. `credit_fraction` is the master switch: the share of each batch from our prior. Mitra (`papers/2025/10_Zhang_Mitra`) finds mixtures beat single priors, so an interior optimum is expected.

In [ ]:
FIGS.save(training_plots.metric_by_lever(TASK, "credit_fraction", exp=EXP), "metric_by_lever_credit_fraction",
    caption="Real-data AUC against training step, one mean line per value of credit fraction, averaged over the arms that share each value.");

## 7. Credit versus out-of-domain retention

The question Exp2 exists to answer: does specialising on credit erode the general ability the released model came with? Purucker 2026 shows TFMs already lose to GBDTs off the IID regime (`papers/2026/06_Purucker_BeyondIID`), but tests ICL only — fine-tuning under shift is the open question this measures directly.

In [ ]:
FIGS.save(training_plots.real_vs_ood(TASK, exp=EXP), "real_vs_ood",
    caption="Mean AUC on the real-credit datasets (solid) and the out-of-domain suites (dashed) against training step, averaged across the fine-tuning arms.");

## 8. Every logged evaluation metric

Not just the headline: every metric the progress-eval records, so a prior that helps ranking (AUC) but hurts calibration (Brier, ECE) is caught. The library notes calibration is a first-class selling point yet under-measured across adaptation regimes (`SYNTHESIS.md`).

In [ ]:
for _p in range(1, training_plots.metric_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.all_eval_metrics(TASK, _p, exp=EXP), f"eval_metrics_p{_p}",
        caption="Each logged real-data evaluation metric, averaged over arms and datasets, against training step; the arrow in each panel title marks the improving direction.");

## 9. Per-dataset learning curves

Averages hide the dataset the prior helps or hurts. One panel per real dataset, credit against control, so a selective effect is visible rather than washed into the mean.

In [ ]:
for _p in range(1, training_plots.per_dataset_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_dataset_curves(TASK, _p, exp=EXP), f"per_dataset_p{_p}",
        caption="Real-data AUC against training step, one panel per evaluation dataset, credit arms (blue) against control arms (grey).");

## 10. Per-configuration training curves

The whole sweep at a glance: every arm's own loss and headline curve in one small panel, so an arm that behaved unlike its neighbours is easy to spot.

In [ ]:
for _p in range(1, training_plots.config_pages(TASK, exp=EXP) + 1):
    FIGS.save(training_plots.per_config(TASK, _p, exp=EXP), f"per_config_p{_p}",
        caption="Per-arm training curves against step: train loss (grey, left axis) and real-data AUC (blue, right axis), one panel per configuration.");

## 11. Final score by lever

Reading the sweep as a screen: every finished arm's final headline score, grouped by each lever in turn, with the group mean marked — the clearest read on which knob actually moves the number.

In [ ]:
FIGS.save(training_plots.final_metric_by_lever(TASK, exp=EXP), "final_metric_by_lever",
    caption="Final real-data AUC of every finished arm as points, one column per swept lever, with a horizontal bar at each group mean.");

## 12. Best versus worst arm

The extremes side by side: the arms that reached the highest and lowest final score, loss and metric together, to see whether the worst arm failed to descend or descended to a worse place.

In [ ]:
FIGS.save(training_plots.best_and_worst(TASK, exp=EXP), "best_and_worst",
    caption="Train loss and real-data AUC against training step for the best and worst arm by final AUC.");

## 13. Hardware during training

Was the machine actually working? TabICL generates its prior on the CPU, so a data-starved run and a compute-bound run look identical from the loss curve but need opposite fixes (`docs/VSC.md`); the utilisation trace tells them apart.

In [ ]:
FIGS.save(training_plots.hardware(TASK, exp=EXP), "hardware",
    caption="GPU utilisation, throughput and peak allocated memory against training step, pooled across arms; the dashed line marks 70 percent utilisation.");

## 14. Per-block gradient flow

Is every part of the model learning? The column encoder, row encoder, ICL blocks and head each get a curve. Under an Exp2 freeze strategy a frozen stack sits on the floor — a result here, not a fault.

In [ ]:
FIGS.save(training_plots.gradient_flow(TASK, exp=EXP), "gradient_flow",
    caption="Mean per-block gradient L2 norm (column encoder, row encoder, ICL blocks, head) against training step on a logarithmic axis.");

## 15. Gradient-to-weight ratio

The interpretable version of the gradient flow: a raw gradient of 0.01 is tiny against weights of 0.1 and enormous against 1e-5, so the ratio is what says whether a block is effectively frozen — and the loss curve never would.

In [ ]:
FIGS.save(training_plots.weight_gradient_ratios(TASK, exp=EXP), "weight_gradient_ratios",
    caption="Mean per-block ratio of gradient norm to weight norm against training step on a logarithmic axis, one line per architecture block.");

## Summary

A printed, copy-pasteable recap of every section, the tfm-library sources this notebook leans on, and the figure inventory.

In [ ]:
print(training_plots.training_summary(TASK, exp=EXP))
print()
print(literature.references_md(["optimizer", "purucker_icl", "calibration_gap", "tanna_resampling", "merton_vasicek"]))
print()
print(FIGS.summary())